<a href="https://colab.research.google.com/github/Decoding-Data-Science/airesidency/blob/main/Copy_of_simple_langchain_rag_colab_demo_11thjul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple LangChain + OpenAI + Chroma RAG Demo in Google Colab

This notebook is a **beginner-friendly, line-by-line demo** for AI residents.

It shows how to:

1. Load API keys from **Google Colab Secrets**
2. Upload a few **PDF files**
3. Read and split the PDFs into chunks
4. Create embeddings with **OpenAI**
5. Store vectors in **ChromaDB**
6. Run **RAG** over the uploaded files
7. Add **web search** using **SERPAPI**
8. Add a very simple **memory** component so the chatbot remembers earlier context

---

## Colab secrets you should create first

In the **Secrets** panel in Colab, create these secrets and turn on notebook access:

- `OPENAI_API_KEY`
- `SERPAPI_API_KEY`

This notebook keeps things intentionally simple and heavily commented so you can teach from it directly.

In [ ]:
# Install the packages we need.
# We keep the list small and focused on the demo.

!pip -q install -U langchain langchain-community langchain-openai langchain-chroma chromadb pypdf google-search-results

## 1) Imports

We import only the core pieces we need:
- file loading
- chunking
- embeddings
- vector database
- chat model
- prompt templates
- simple memory
- SERPAPI search

In [ ]:
import os
import shutil
from pathlib import Path

from google.colab import files, userdata

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser

from langchain_community.utilities import SerpAPIWrapper

/tmp/ipykernel_17085/2559511183.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


## 2) Load API keys from Google Colab secrets

This is cleaner than pasting keys directly into the notebook.

In [ ]:
# Read secrets from the Colab Secrets manager and place them in environment variables.
# Make sure you created both secrets in the left-side Secrets panel.

os.environ["OPENAI_API_KEY"] = userdata.get("openai")
os.environ["SERPAPI_API_KEY"] = userdata.get("SERPAPI_API_KEY")

print("OPENAI key loaded:", bool(os.environ.get("OPENAI_API_KEY")))
print("SERPAPI key loaded:", bool(os.environ.get("SERPAPI_API_KEY")))

OPENAI key loaded: True
SERPAPI key loaded: True


## 3) Create a folder and upload PDFs

This cell opens a file picker in Colab.
Upload a few PDFs and we will save them into `/content/data`.

You can re-run this cell any time to add more files.

In [ ]:
DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()

for file_name, file_bytes in uploaded.items():
    save_path = DATA_DIR / file_name
    with open(save_path, "wb") as f:
        f.write(file_bytes)

print("Files currently in data folder:")
for p in DATA_DIR.iterdir():
    print("-", p.name)

Saving DDS_Employee_Handbook_Synthetic_v1.pdf to DDS_Employee_Handbook_Synthetic_v1 (2).pdf
Saving DDS_HR_FAQ_Synthetic_v1.pdf to DDS_HR_FAQ_Synthetic_v1 (2).pdf
Saving DDS_Leave_Policy_v2.pdf to DDS_Leave_Policy_v2 (2).pdf
Saving DDS_Remote_Work_Policy_Synthetic_v1.pdf to DDS_Remote_Work_Policy_Synthetic_v1 (2).pdf
Files currently in data folder:
- DDS_Remote_Work_Policy_Synthetic_v1.pdf
- DDS_Employee_Handbook_Synthetic_v1 (2).pdf
- DDS_Employee_Handbook_Synthetic_v1.pdf
- DDS_HR_FAQ_Synthetic_v1.pdf
- DDS_Remote_Work_Policy_Synthetic_v1 (2).pdf
- DDS_Remote_Work_Policy_Synthetic_v1 (1).pdf
- DDS_Leave_Policy_v2 (1).pdf
- DDS_Leave_Policy_v2.pdf
- DDS_HR_FAQ_Synthetic_v1 (2).pdf
- .ipynb_checkpoints
- DDS_Leave_Policy_v2 (2).pdf


## 4) Load the PDF files

`PyPDFDirectoryLoader` reads all PDFs inside a folder.

Each page usually becomes a `Document` object with text plus metadata.

In [ ]:
loader = PyPDFDirectoryLoader(str(DATA_DIR))
docs = loader.load()

print("Number of loaded document pages:", len(docs))
print("\nSample metadata from the first loaded page:")
print(docs[0].metadata if docs else "No documents found.")

Number of loaded document pages: 34

Sample metadata from the first loaded page:
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-03-03T08:04:57+00:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-03-03T08:04:57+00:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': '/content/data/DDS_Remote_Work_Policy_Synthetic_v1.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


## 5) Split the documents into chunks

LLMs and vector stores usually work better on smaller chunks than on entire PDF pages.

We use `RecursiveCharacterTextSplitter`, which is a common default choice.
- `chunk_size=1000` means each chunk is about 1000 characters
- `chunk_overlap=200` keeps some repeated context between chunks

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = text_splitter.split_documents(docs)

print("Number of chunks:", len(chunks))
print("\nPreview of the first chunk:")
print(chunks[0].page_content[:800] if chunks else "No chunks found.")

Number of chunks: 86

Preview of the first chunk:
DDS Remote Work Policy (Synthetic) v1
Effective date: March 03, 2026  Dubai (GST)
Note: This document is a synthetic, training-friendly employee handbook for demos, onboarding
simulations, and HR-policy chatbot prototypes. It is not legal advice and must be reviewed by qualified
counsel before any real-world use.
1. Remote Work Philosophy
DDS supports remote and hybrid work because our team and learners are global. Remote work is a privilege
tied to trust, outcomes, and security.
Remote success requires:
 Clear goals and ownership.
 High-quality written communication.
 Reliable availability during core collaboration hours.
2. Eligibility & Approval
Remote work depends on role suitability, performance, and operational needs.
 Approval is granted by the line manager.
 HR may define rol


## 6) Create embeddings and store them in ChromaDB

We use:
- `OpenAIEmbeddings` to convert text chunks into vectors
- `Chroma` to store those vectors locally on disk

The `persist_directory` lets us save the vector database so we can reuse it later.

In [ ]:
# If you want a clean rebuild every time you run the notebook, remove the old database folder first.
CHROMA_DIR = "/content/chroma_db"

if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma(
    collection_name="demo_rag_collection",
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
)

# Add the chunks to the vector database.
vectorstore.add_documents(chunks)

print("Chunks indexed into Chroma.")
print("Chroma data saved at:", CHROMA_DIR)

Chunks indexed into Chroma.
Chroma data saved at: /content/chroma_db


## 7) Run a simple similarity search

Before building the full RAG flow, it is useful to show students how retrieval works directly.

In [ ]:
query = "What are the most important topics in these documents?"
results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"\n--- Retrieved chunk {i} ---")
    print("Metadata:", doc.metadata)
    print(doc.page_content[:700])


--- Retrieved chunk 1 ---
Metadata: {'keywords': '', 'moddate': '2026-03-03T08:04:57+00:00', 'trapped': '/False', 'subject': 'unspecified', 'source': '/content/data/DDS_Remote_Work_Policy_Synthetic_v1.pdf', 'total_pages': 2, 'producer': 'ReportLab PDF Library - (opensource)', 'page': 0, 'title': 'untitled', 'author': 'anonymous', 'page_label': '1', 'creationdate': '2026-03-03T08:04:57+00:00', 'creator': 'anonymous'}
 Enable strong authentication (MFA) on all work accounts.
 Keep devices updated with security patches.
 Do not store DDS data on personal cloud drives or unapproved services.
 Report lost devices or suspected compromise immediately.
AI tools:
 Do not paste confidential documents into public AI tools.
 Use approved enterprise environments for sensitive data when available.
6. Meetings, Documentation & Asynchronous Work
DDS prefers async-first documentation:
 Write decisions and action items in shared docs.
 Record key meetings when appropriate (with consent).
 Use 

## 8) Create a retriever

A retriever is the part that fetches the most relevant chunks for a user question.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

## 9) Create the chat model

We use a simple OpenAI chat model for the answer generation step.

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

## 10) Build a simple RAG prompt

The prompt uses:
- the retrieved document context
- chat history
- the user's current question

This lets us demonstrate both **RAG** and **memory** in one place.

In [ ]:
rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a helpful assistant for a beginner RAG demo.

Use the retrieved context to answer the user's question.
If the answer is not in the context, say that clearly.
Keep the answer simple and grounded in the retrieved context.
""",
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    (
        "human",
        """
Question:
{input}

Retrieved context:
{context}
""",
    ),
])

## 11) Helper function to format retrieved documents

This converts the retrieved chunks into one text block for the prompt.

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

## 12) Create a simple RAG chain with memory

We will:
1. retrieve relevant chunks
2. format them
3. pass them to the prompt
4. get the final answer from the LLM

Then we wrap the chain with `RunnableWithMessageHistory` so it remembers prior turns.

This is a very simple in-memory demo.  
It will remember context **while the notebook session is alive**.

In [ ]:
from operator import itemgetter
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Step A: Build a small chain that first retrieves relevant chunks.
retrieval_step = {
    "input": itemgetter("input"),
    "chat_history": itemgetter("chat_history"),
    "context": itemgetter("input") | retriever | RunnableLambda(format_docs),
}

# Step B: Send everything into the prompt, model, and string output parser.
rag_chain = retrieval_step | rag_prompt | llm | StrOutputParser()

# Step C: Store chat history in a Python dictionary.
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Step D: Wrap the chain with message history.
rag_chain_with_memory = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


## 13) Ask questions and see memory in action

Try asking:
- a question about the uploaded PDFs
- then a follow-up question like "Can you summarize that in 3 bullets?"
- then "What was my previous question about?"

Because we attached memory, the model can use the prior conversation.

In [ ]:
session_id = "demo-user-1"

question_1 = "Give me a simple summary of the uploaded documents."
answer_1 = rag_chain_with_memory.invoke(
    {"input": question_1},
    config={"configurable": {"session_id": session_id}}
)

print("Q1:", question_1)
print("\nA1:", answer_1)

Q1: Give me a simple summary of the uploaded documents.

A1: - **Security Measures**: Enable multi-factor authentication (MFA), keep devices updated, and avoid storing sensitive data on personal cloud services.
- **AI Tool Usage**: Do not share confidential documents in public AI tools and use approved environments for sensitive data.
- **Documentation Practices**: Focus on asynchronous documentation, record key meetings with consent, and minimize unnecessary meetings by prioritizing written updates.


In [ ]:
question_2 = "Now give me that summary a above in 3 short bullet points."
answer_2 = rag_chain_with_memory.invoke(
    {"input": question_2},
    config={"configurable": {"session_id": session_id}}
)

print("Q2:", question_2)
print("\nA2:", answer_2)

Q2: Now give me that summary a above in 3 short bullet points.

A2: - **Core Values**: Focus on bias for action, respect and inclusion, data integrity, and a culture of continuous learning.
- **Employment Types**: Team members can be full-time, part-time, or contractors, with details specified in their contracts.
- **Commitment to Equality**: DDS promotes a discrimination-free workplace and encourages reporting of harassment confidentially.


In [ ]:
question_2 = "was there mention anything about leave."
answer_2 = rag_chain_with_memory.invoke(
    {"input": question_2},
    config={"configurable": {"session_id": session_id}}
)

print("Q2:", question_2)
print("\nA2:", answer_2)

Q2: was there mention anything about leave.

A2: Yes, the retrieved context mentions leave. Here are the key points:

- Full-time employees are entitled to 22 working days of paid annual leave per year, while part-time employees receive a pro-rata entitlement.
- Annual leave accrues at a rate of 1.83 working days per completed calendar month of service.
- The maximum single block of annual leave is 15 consecutive working days, and at least 5 days must be taken as one continuous block each year.


In [ ]:
# This lets students inspect exactly what was saved in memory.
history = get_session_history(session_id)

print("Stored messages in memory:")
for msg in history.messages:
    print(f"\n{msg.type.upper()}:")
    print(msg.content)

Stored messages in memory:

HUMAN:
Now give me that summary a above in 3 short bullet points.

AI:
- **Core Values**: Emphasize bias for action, respect and inclusion, data integrity, and a learning culture.
- **Employment Types**: Team members can be full-time, part-time, or contractors, with details defined in offer letters/contracts.
- **Workplace Commitment**: DDS ensures a discrimination-free environment and encourages reporting of harassment confidentially.

HUMAN:
Now give me that summary a above in 3 short bullet points.

AI:
- **Core Values**: Focus on bias for action, respect and inclusion, data integrity, and a culture of continuous learning.
- **Employment Types**: Team members can be full-time, part-time, or contractors, with details specified in their contracts.
- **Commitment to Equality**: DDS promotes a discrimination-free workplace and encourages reporting of harassment confidentially.

HUMAN:
was there mention anything about leave.

AI:
Yes, the retrieved context men

## 14) Add web search with SERPAPI

Now we add a simple search tool.

This is useful when:
- the uploaded PDFs do not contain the answer
- the user asks for something recent or external
- you want to compare internal document answers with web results

In [ ]:
search = SerpAPIWrapper()

web_result = search.run("Latest news about LangChain")
print(web_result[:2000])

["LangChain and NVIDIA launch the NemoClaw Deep Agents Blueprint. The LangChain Team. July 8, 2026 ; Your coding agent bill doubled. Here's how to fix it. Amy Ru.", "What's new in LangChain v1 · Main loop integration: Structured output is now generated in the main loop instead of requiring an additional LLM call · Parsing ...", "In this video, I cover LangChain's recent announcements about raising money at over a $1.25 valuation and the launch of LangChain and ...", 'Everyone was a fan of LangChain until a few months ago. Plenty of YT tutorials, Blogs, Reels but suddenly everyone is bashing them.', 'LangChain provides the engineering platform and open source frameworks developers use to build, test, and deploy reliable AI agents.', 'LangChain is a framework for building agents and LLM-powered applications. It helps you chain together interoperable components and third-party integrations ...', 'LangChain itself blows my mind as one of the most useless libraries to exist. I hope this doe

## 15) A very simple hybrid function: use docs + optional web search

This helper does the following:
- runs retrieval on the local PDFs
- optionally runs SERPAPI search
- combines both into one prompt
- keeps memory

This is still intentionally simple and easy to teach.

In [ ]:
hybrid_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a helpful assistant in a simple beginner demo.

Use the local document context first.
Use web search results only as an extra source when they are provided.
If something is missing from the documents, say so clearly.
""",
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    (
        "human",
        """
Question:
{input}

Local document context:
{context}

Optional web search results:
{web_context}
""",
    ),
])

hybrid_chain = (
    {
        "input": itemgetter("input"),
        "chat_history": itemgetter("chat_history"),
        "context": itemgetter("input") | retriever | RunnableLambda(format_docs),
        "web_context": itemgetter("web_context"),
    }
    | hybrid_prompt
    | llm
    | StrOutputParser()
)

hybrid_chain_with_memory = RunnableWithMessageHistory(
    hybrid_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
def ask_hybrid(question, use_web=False, session_id="demo-user-2"):
    web_context = ""

    if use_web:
        web_context = search.run(question)

    answer = hybrid_chain_with_memory.invoke(
        {
            "input": question,
            "web_context": web_context,
        },
        config={"configurable": {"session_id": session_id}},
    )

    return answer

## 16) Test the hybrid assistant

Use `use_web=False` for pure document RAG.  
Use `use_web=True` when you want external search results included.

In [ ]:
print(ask_hybrid("What are the main ideas discussed in my uploaded PDFs?", use_web=False))

The main ideas discussed in your uploaded PDFs include:

- **Security Practices**: Emphasize the importance of enabling multi-factor authentication (MFA), keeping devices updated with security patches, and avoiding the storage of DDS data on personal or unapproved cloud services. Immediate reporting of lost devices or suspected security compromises is also highlighted.

- **Use of AI Tools**: Guidelines are provided for handling confidential documents, advising against pasting them into public AI tools and recommending the use of approved enterprise environments for sensitive data.

- **Meetings and Documentation**: DDS promotes an asynchronous-first approach to documentation, encouraging the use of shared documents for decisions and action items, recording key meetings with consent, and utilizing clear templates for project updates. There is also a focus on reducing unnecessary meetings and prioritizing written updates.


In [ ]:
print(ask_hybrid("What are some recent updates about Decoding Data Science?", use_web=True))

Recent updates about Decoding Data Science (DDS) include:

- **Data Protection & Privacy**: DDS emphasizes responsible handling of personal and business data, including enabling multi-factor authentication (MFA), keeping devices updated, and avoiding the storage of DDS data on personal or unapproved cloud services.

- **Use of AI Tools**: There are guidelines against pasting confidential documents into public AI tools and a recommendation to use approved enterprise environments for sensitive data.

- **Meetings and Documentation**: DDS promotes an asynchronous-first approach to documentation, encouraging the use of shared documents for decisions and action items, recording key meetings with consent, and reducing unnecessary meetings in favor of written updates. 

These updates reflect DDS's commitment to security, effective communication, and efficient work practices.


In [ ]:
print(ask_hybrid("What is langchain?", use_web=True))

LangChain is an open-source framework designed for building applications that utilize large language models (LLMs). Here are some key points about LangChain:

- **Integration**: It connects LLMs with various data sources and tools, allowing developers to create more complex and capable AI applications.

- **Pre-built Architectures**: LangChain offers pre-built agent architectures and integrations, making it easier for developers to start building AI agents quickly.

- **Orchestration Framework**: It serves as an orchestration framework that simplifies the development, testing, and deployment of applications using LLMs, such as chatbots and virtual assistants.

Overall, LangChain provides the necessary tools and modules to facilitate the creation of AI applications that leverage the power of large language models.


## 17) Save and reload the vector database later

Because we used a `persist_directory`, we can reconnect to the same Chroma database later without rebuilding it from scratch.

This is helpful in Colab demos when you want to explain indexing once, then reuse the stored vectors.

In [ ]:
reloaded_vectorstore = Chroma(
    collection_name="demo_rag_collection",
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
)

reloaded_results = reloaded_vectorstore.similarity_search(
    "Give me a summary of the documents",
    k=2
)

for i, doc in enumerate(reloaded_results, start=1):
    print(f"\nReloaded result {i}:")
    print(doc.page_content[:600])


Reloaded result 1:
 Enable strong authentication (MFA) on all work accounts.
 Keep devices updated with security patches.
 Do not store DDS data on personal cloud drives or unapproved services.
 Report lost devices or suspected compromise immediately.
AI tools:
 Do not paste confidential documents into public AI tools.
 Use approved enterprise environments for sensitive data when available.
6. Meetings, Documentation & Asynchronous Work
DDS prefers async-first documentation:
 Write decisions and action items in shared docs.
 Record key meetings when appropriate (with consent).
 Use clear templates for p

Reloaded result 2:
 Enable strong authentication (MFA) on all work accounts.
 Keep devices updated with security patches.
 Do not store DDS data on personal cloud drives or unapproved services.
 Report lost devices or suspected compromise immediately.
AI tools:
 Do not paste confidential documents into public AI tools.
 Use approved enterprise environments for sensitive

## 18) Teaching notes

### What this notebook demonstrates
- **LangChain loaders** for PDFs
- **Chunking**
- **OpenAI embeddings**
- **Chroma vector database**
- **Retriever-based RAG**
- **SERPAPI search**
- **Simple memory with chat history**

### What this notebook does *not* try to do
- advanced agents
- tool calling orchestration
- citations with structured sources
- production-grade memory
- evaluation
- UI deployment

That is intentional. This notebook is meant to be the **smallest clean teaching demo**.

## 19) Suggested next upgrades for residents

After students understand this notebook, the next upgrades can be:
1. Add a small **Gradio** or **Streamlit** UI
2. Add **source citations**
3. Add support for more file types
4. Add **query rewriting**
5. Add **conversation-aware retrieval**
6. Add **LangSmith tracing**
7. Add **better memory storage** beyond notebook runtime